NOTE: Kal humne p-value simulation (5000 shuffles) se nikaala tha. t-test wahi kaam ek formula se seedha kar deta hai, bina simulation ke. Yeh do groups ke means ka fark check karta hai, aur fark ko us fark ki "noise" (standard error) se compare karta hai:

t = (mean₁ - mean₂) / standard error

t jitna bada (positive ya negative), p-value utna chhota.

1. Data load aur groups banao:

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv('../data/processed/customer_churn_labeled.csv')

churned = df[df['Churned'] == 1]['Monetary']
not_churned = df[df['Churned'] == 0]['Monetary']

print(f"Churned: n={len(churned)}, mean={churned.mean():.2f}")
print(f"Not churned: n={len(not_churned)}, mean={not_churned.mean():.2f}")

Churned: n=2987, mean=1133.22
Not churned: n=2894, mean=4834.09


2. Independent two-sample t-test (Welch's version):

In [2]:
t_stat, p_value = stats.ttest_ind(churned, not_churned, equal_var=False)
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.6f}")

alpha = 0.05
if p_value < alpha:
    print("H0 reject: Monetary spend ka fark statistically significant hai.")
else:
    print("H0 reject nahi kar sakte: fark chance se ho sakta hai.")

t-statistic: -9.7604
p-value: 0.000000
H0 reject: Monetary spend ka fark statistically significant hai.


Note: equal_var=False ka matlab Welch's t-test, jo maanta hai ki dono groups ka variance alag ho sakta hai. Yeh real data ke liye safer default hai, kyunki churned aur non-churned groups ka spread aksar alag hota hai. Kal ke permutation p-value se isko compare karo, dono ka conclusion same hona chahiye.

3. Effect size (Cohen's d), sirf p-value kaafi nahi hota (kal ki galatfehmi #2):

In [3]:
n1, n2 = len(churned), len(not_churned)
s1, s2 = churned.std(), not_churned.std()
pooled_std = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
cohens_d = (churned.mean() - not_churned.mean()) / pooled_std
print(f"Cohen's d: {cohens_d:.3f}")

Cohen's d: -0.258


Rule of thumb: |d| ≈ 0.2 chhota, 0.5 medium, 0.8+ bada effect. Bade sample mein p-value bahut chhota aa sakta hai jabki d chhota ho, matlab fark "real" hai lekin business ke liye bahut bada nahi.

4. Difference ka 95% Confidence Interval:

In [4]:
diff = churned.mean() - not_churned.mean()
se = np.sqrt(s1**2 / n1 + s2**2 / n2)
ci_low, ci_high = diff - 1.96 * se, diff + 1.96 * se
print(f"Mean difference: {diff:.2f}")
print(f"95% CI: ({ci_low:.2f}, {ci_high:.2f})")

Mean difference: -3700.87
95% CI: (-4444.04, -2957.70)


Agar CI mein 0 shamil nahi hai, to yeh p < 0.05 wale conclusion se match karta hai.

5. Assumption check: Monetary skewed hai (Day 42), to t-test par kitna bharosa?

In [5]:
print(f"Skewness churned: {churned.skew():.2f}")
print(f"Skewness not churned: {not_churned.skew():.2f}")

Skewness churned: 13.91
Skewness not churned: 18.38


t-test mean par kaam karta hai aur outliers se asar leta hai. Bade sample (hamare paas hazaron customers hain) mein Central Limit Theorem se result kaafi had tak stable rehta hai, lekin heavy skew mein do backup tests chalana achhi practice hai:

6. Backup 1: Log-transform ke baad t-test (Day 42 ka trick):

In [6]:
log_churned = np.log1p(churned)
log_not_churned = np.log1p(not_churned)

t_log, p_log = stats.ttest_ind(log_churned, log_not_churned, equal_var=False)
print(f"Log-transformed t-test: t={t_log:.4f}, p={p_log:.6f}")

Log-transformed t-test: t=-35.4436, p=0.000000


7. Backup 2: Mann-Whitney U test (non-parametric, mean ki jagah distributions compare karta hai):

In [7]:
u_stat, p_mw = stats.mannwhitneyu(churned, not_churned, alternative='two-sided')
print(f"Mann-Whitney U: U={u_stat:.0f}, p={p_mw:.6f}")

Mann-Whitney U: U=2172836, p=0.000000


Agar teeno tests (Welch, log-t-test, Mann-Whitney) same direction mein significant aayein, to conclusion robust hai.

8. One-sample t-test: kya average Frequency kisi given value se alag hai?

In [8]:
t1, p1 = stats.ttest_1samp(df['Frequency'], popmean=5)
print(f"Sample mean Frequency: {df['Frequency'].mean():.2f}")
print(f"One-sample t-test vs 5: t={t1:.4f}, p={p1:.6f}")

Sample mean Frequency: 6.29
One-sample t-test vs 5: t=7.5857, p=0.000000


9. Frequency par bhi same test (Day 44 ke practice wale sawal ka formal version):

In [9]:
t_f, p_f = stats.ttest_ind(
    df[df['Churned'] == 1]['Frequency'],
    df[df['Churned'] == 0]['Frequency'],
    equal_var=False
)
print(f"Frequency t-test: t={t_f:.4f}, p={p_f:.6f}")

Frequency t-test: t=-19.7582, p=0.000000


Practice questions:

1. humne notebook mein ek chhota markdown cell likho: H0, H1, test ka naam, p-value, Cohen's d, aur conclusion (2-3 lines, business language mein). Yeh format interview mein bhi kaam aayega.

### A/B Testing / Hypothesis Testing Summary: Monetary Spend vs Churn

* **Hypotheses (H0 & H1):** 
  - **H0 (Null Hypothesis):** Churned aur non-churned customers ke average monetary spend mein koi farq nahi hai.
  - **H1 (Alternative Hypothesis):** Churned aur non-churned customers ke average monetary spend mein farq hai.
* **Test Details:** Welch's Independent Two-Sample t-test (`equal_var=False`), jiske sath Cohen's d effect size aur Mann-Whitney U backup test bhi run kiya gaya.
* **Results:** p-value < 0.05 aane par H0 reject kiya gaya, aur Cohen's d se spend ke farq ka magnitude (effect size) dekha gaya.
* **Business Conclusion:** Churned aur non-churned customers ki spending habits mein statistical significance hai. Iska matlab hai ki customer value (monetary spend) churn prediction aur retention strategies ke liye ek mahatvapurna factor hai.
*

2. Recency par t-test chalao aur likho ki result "obvious" kyun hai (churn ki definition hi Recency par based hai, is liye test se naya insight nahi milta).

- Agar aap Recency par t-test chalayenge (df[df['Churned'] == 1]['Recency'] vs df[df['Churned'] == 0]['Recency']), toh p-value extremely chhota (zero ke kareeb) aayega aur t-statistic bahut bada hoga.


- Yeh "obvious" ya redundant kyun hai?
    - Kyunki zyadatar customer churn models ya business definitions mein "Churn" ki paribhasha hi is baat par aadharit hoti hai ki customer ne pichle kuch dino (Recency) se kharidari nahi ki hai. Jab churn ka khud ka base hi high recency (lamba samay bina purchase ke) par tikha ho, toh recency aur churned status mein strong mathematical link pehle se hi mojood hota hai. Isliye, is test se koi naya hidden insight nahi milta, yeh bas ek known definition ka statistical confirmation hota hai.